In [ ]:
# hide
# no-output
from IPython.utils.capture import capture_output
with capture_output():
    %pip install -q plotly anywidget

import asyncio
import os
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import Audio
import pyquist as pq
import icm_plotly
from icm_plotly import RED, BLUE, GOLD, IRON, TEAL, STEEL

Place a sustain loop $[t_s, t_e)$ on the clarinet recording. The top plot
shades the loop (orange) and marks where every note jumps to the release (dotted purple). The
bottom plot zooms in on a wrap: the end of the loop jumps back to its start
(purple), against what the recording would have played next (dashed gray).
The audio card plays a 5 second note. Try a loop of exactly one period, then
several hundred audio samples, then add a crossfade.

In [ ]:
# hide
# autorun
START0 = 0.586803                   # the chapter's loop: audio samples 25,878...
PERIODS0 = 7943 / (44100 / 932.33)  # ...to 33,821, about 167.9 periods
FADE0 = 0.0                         # no crossfade to start

clarinet = pq.Audio.from_file("../assets/clarinet.wav").as_mono()
SR = clarinet.sample_rate
X = np.asarray(clarinet.samples).reshape(-1)
X = X / np.abs(X).max()
P = SR / 932.33                     # one period of A#5: about 47.30 audio samples
RELEASE = round(3.321361 * SR)      # where every note jumps to the release
T_NOTE = 5.0                        # the note the audio card plays, in seconds
W = round(0.006 * SR)               # the seam view: 6 ms on each side
PURPLE = "#6E3B87"


def loop_points(start, periods, P=P, SR=SR, RELEASE=RELEASE):
    s = round(start * SR)
    e = min(s + max(1, round(periods * P)), RELEASE)
    return s, e


def one_pass(s, e, fade_ms, X=X, SR=SR):
    """One pass of the loop. With a crossfade, its last C samples fade out
    while the audio that led into s fades in, so the wrap back to s is seamless."""
    loop = X[s:e].copy()
    C = min(round(fade_ms / 1000 * SR), e - s, s)
    if C > 0:
        ramp = np.linspace(0, 1, C)
        loop[-C:] = (1 - ramp) * X[e - C:e] + ramp * X[s - C:s]
    return loop


def seam(s, e, fade_ms, one_pass=one_pass, X=X, W=W):
    """What plays around a wrap, and what the recording would have played."""
    loop = one_pass(s, e, fade_ms)
    before = np.concatenate([X[:s], loop])[-W:]       # end of a pass...
    after = X[s:s + W]                                 # ...then back to s
    return np.concatenate([before, after]), X[e - W:e + W]


def note(s, e, fade_ms, one_pass=one_pass, X=X, SR=SR, RELEASE=RELEASE,
         T_NOTE=T_NOTE):
    loop = one_pass(s, e, fade_ms)
    n = max(1, round((T_NOTE * SR - s - (len(X) - RELEASE)) / len(loop)))
    return np.concatenate([X[:s], np.tile(loop, n), X[RELEASE:]])


def envelope(x, dt, n_blocks=600):
    """Times and min/max values of x in n_blocks blocks, so peaks survive."""
    step = max(1, len(x) // n_blocks)
    n = len(x) // step * step
    blocks = x[:n].reshape(-1, step)
    t = (np.arange(n // step) + 0.5) * step * dt
    return np.repeat(t, 2), np.column_stack([blocks.min(1), blocks.max(1)]).ravel()


T_ENV, E_ENV = envelope(X, 1 / SR)
TS = (np.arange(2 * W) - W) / SR * 1000          # seam view time axis, in ms


def band(a, b):
    return [a, a, b, b, a], [-1.1, 1.1, 1.1, -1.1, -1.1]


def figure():
    s, e = loop_points(START0, PERIODS0)
    played, natural = seam(s, e, FADE0)
    fig = make_subplots(rows=2, cols=1, row_heights=[0.36, 0.64],
                        vertical_spacing=0.2)
    bx, by = band(s / SR, e / SR)
    fig.add_scatter(x=bx, y=by, fill="toself", fillcolor="rgba(255,127,14,0.4)",
                    line=dict(width=0), hoverinfo="skip", row=1, col=1)
    fig.add_scatter(x=T_ENV, y=E_ENV, mode="lines",
                    line=dict(color=IRON, width=1), row=1, col=1)
    fig.add_scatter(x=[RELEASE / SR] * 2, y=[-1.1, 1.1], mode="lines",
                    line=dict(color=PURPLE, width=1.5, dash="dot"), row=1, col=1)
    fig.add_scatter(x=[0, 0], y=[-1.1, 1.1], mode="lines",
                    line=dict(color=RED, width=1.2, dash="dot"), row=2, col=1)
    fig.add_scatter(x=TS, y=natural, mode="lines",
                    line=dict(color=STEEL, width=1.6, dash="dash"), row=2, col=1)
    fig.add_scatter(x=TS, y=played, mode="lines",
                    line=dict(color=PURPLE, width=2), row=2, col=1)
    fig.update_xaxes(range=[0, len(X) / SR], title_text="Time in the recording (s)",
                     fixedrange=True, row=1, col=1)
    fig.update_yaxes(range=[-1.1, 1.1], showticklabels=False, fixedrange=True,
                     row=1, col=1)
    fig.update_xaxes(range=[TS[0], TS[-1]],
                     title_text="Time around the wrap (ms)", fixedrange=True,
                     row=2, col=1)
    fig.update_yaxes(range=[-1.1, 1.1], title_text="Amplitude",
                     fixedrange=True, row=2, col=1)
    fig.update_layout(height=460)
    return fig


def controls(fig):
    start = widgets.FloatSlider(description="Loop start $t_s$ (s)", min=0.3,
                                max=3.0, step=0.0001, value=START0,
                                readout_format=".4f")
    periods = widgets.FloatSlider(description="Loop length (periods)", min=1.0,
                                  max=200.0, step=0.01, value=PERIODS0,
                                  readout_format=".2f")
    fade = widgets.FloatSlider(description="Crossfade (ms)", min=0.0, max=5.0,
                               step=0.1, value=FADE0, readout_format=".1f")
    readout = widgets.HTML()

    # the defaults snapshot the arrays; the page's notebooks share one kernel
    def update(start, periods, fade, loop_points=loop_points, seam=seam,
               band=band, SR=SR, P=P, W=W, readout=readout):
        s, e = loop_points(start, periods)
        played, natural = seam(s, e, fade)
        bx, by = band(s / SR, e / SR)
        with fig.batch_update():
            fig.data[0].x = bx
            fig.data[4].y = natural
            fig.data[5].y = played
        L = e - s
        m = max(1, round(L / P))            # whole periods the loop nearly spans
        frac = L / P - m                    # how far off each wrap lands
        steps = np.abs(np.diff(natural))
        jump = abs(played[W] - played[W - 1]) / steps.max()
        text = (f"<i>t</i><sub>e</sub> &minus; <i>t</i><sub>s</sub> = {L:,} "
                f"audio samples = {L / P:.2f} periods "
                f"&nbsp;·&nbsp; each wrap shifts the waveform by "
                f"{f'{frac:+.2f}'.replace('-', '&minus;')} "
                f"of a period")
        if abs(frac) < 0.25:
            cents = -1200 * np.log2(L / (m * P))
            text += f" ({f'{cents:+.1f}'.replace('-', '&minus;')} cents out of tune)"
        if jump > 1.5:
            text += (f" &nbsp;·&nbsp; the step across the wrap is {jump:.1f}&times; "
                     f"the largest step nearby: a click")
        else:
            text += " &nbsp;·&nbsp; the wrap itself is smooth"
        readout.value = f"<span style='font-size:0.9em'>{text}</span>"

    widgets.interactive_output(update, {"start": start, "periods": periods,
                                        "fade": fade})

    # the audio card under the controls: the previous clip stays in place
    # while you drag (so the layout never jumps) and is swapped for the new
    # one when the pointer releases (keyboard nudges settle on a timer). It is
    # written through the Output's synced `outputs` trait, which works
    # outside a kernel message, where display() output has no destination
    out = widgets.Output()
    gate = icm_plotly.release_gate()   # pointer state: is a slider mid-drag?
    pending = []
    dirty = []

    def show_audio(x, SR=SR):
        audio = Audio(x.astype(np.float32), rate=SR, normalize=False)
        data, metadata = get_ipython().display_formatter.format(audio)
        # one assignment swaps the old card for the new one in place, so
        # the page never shows an empty card and nothing shifts
        out.outputs = ({"output_type": "display_data",
                        "data": data, "metadata": metadata},)

    def render(loop_points=loop_points, note=note):
        s, e = loop_points(start.value, periods.value)
        y = 0.5 * note(s, e, fade.value)
        y[-441:] *= np.linspace(1, 0, 441)
        show_audio(y)

    async def settle():
        await asyncio.sleep(0.25)
        pending.clear()
        if dirty and not gate.dragging:
            dirty.clear()
            render()

    def on_change(_):
        dirty.append(True)
        if pending:
            pending.pop().cancel()
        pending.append(asyncio.ensure_future(settle()))

    def on_release(change):
        if not change["new"] and dirty:
            if pending:
                pending.pop().cancel()
            dirty.clear()
            render()

    gate.observe(on_release, names="dragging")

    for s in (start, periods, fade):
        s.observe(on_change, names="value")
    if not os.environ.get("ICM_BOOK_BUILD"):   # the build bakes no card
        render()
    return widgets.VBox([start, periods, fade, readout, out, gate])

icm_plotly.show(figure, controls)